# Hybrid Yelp Search + Recommendation (TensorFlow / TFRS)

This notebook runs the **full pipeline**:

1. Load + preprocess Yelp Open Dataset (business + reviews)
2. Train **Stage 1 Retrieval** (two-tower model)
3. Train **Stage 2 Ranking** (reranker)
4. Build a **HybridEngine** for search / recommendation / hybrid queries
5. Evaluate **Recall@K** and **NDCG@K**
6. (Optional) Add **CLIP image embeddings** (offline precompute)

Project code lives under `modeling/` and `scripts/`.


## 0) Setup

Run this notebook from the repository root (where `modeling/` exists).

Expected dataset files:
- `data/yelp/yelp_academic_dataset_business.json`
- `data/yelp/yelp_academic_dataset_review.json`

Optional (for CLIP):
- `data/yelp/yelp_academic_dataset_photo.json`
- Local folder with photo images named by `photo_id`.


In [ ]:
# If needed, install deps (uncomment)
# !pip install -r requirements.txt

import os
import numpy as np
import pandas as pd
import tensorflow as tf

print('TF version:', tf.__version__)


## 1) Load config + set seeds

Edit `configs/config.yaml` to change city/category filters, pruning thresholds, model sizes, and training epochs.


In [ ]:
from modeling.config_utils import load_config, set_global_seeds

CFG_PATH = 'configs/config.yaml'
cfg = load_config(CFG_PATH)
set_global_seeds(int(cfg['experiment']['seed']))

cfg

## 2) Load Yelp + preprocess

This applies standard recommender preprocessing:
- optional region/category filtering
- prune low-activity users and businesses
- build implicit interactions with engagement weight
- leave-last-out split


In [ ]:
from modeling.data_pipeline import load_yelp, preprocess, leave_last_out

business_df, review_df = load_yelp(cfg['data']['root'])
business_df, inter = preprocess(
    business_df, review_df,
    state=cfg['data']['state'],
    city=cfg['data']['city'],
    category_contains=cfg['data']['category_contains'],
    min_biz_reviews=int(cfg['data']['min_biz_reviews']),
    min_user_reviews=int(cfg['data']['min_user_reviews']),
)

train_df, test_df = leave_last_out(inter)

print('Businesses:', len(business_df))
print('Train interactions:', len(train_df))
print('Test interactions:', len(test_df))
train_df.head()

## 3) Optional: Load CLIP image embeddings

If you precomputed per-business CLIP embeddings via:

```bash
python scripts/compute_clip_embeddings.py \
  --photo_json data/yelp/yelp_academic_dataset_photo.json \
  --images_dir /path/to/yelp_photos \
  --out_csv outputs/business_image_embeddings.csv
```

…then set `features.use_image_embeddings: true` in `configs/config.yaml` and load the CSV below.

If you don't have image embeddings, leave this cell as-is.


In [ ]:
from modeling.data_pipeline import load_business_image_embeddings

USE_IMAGE = bool(cfg['features']['use_image_embeddings'])
IMAGE_EMB_PATH = None  # e.g., 'outputs/business_image_embeddings.csv'

img_df = None
if USE_IMAGE:
    if IMAGE_EMB_PATH is None:
        raise ValueError('Set IMAGE_EMB_PATH to the CSV/Parquet containing image_emb_0.. columns.')
    img_df = load_business_image_embeddings(IMAGE_EMB_PATH)
    print('Loaded image embeddings for businesses:', len(img_df))
else:
    print('Image embeddings disabled (cfg.features.use_image_embeddings=false)')


## 4) Build TensorFlow datasets

Creates:
- `train_ds`: interactions for retrieval training
- `cand_ds`: business corpus for text adaptation and indexing
- `rank_ds`: training data for the ranking model


In [ ]:
from train import build_tf_datasets, build_rank_ds

batch_size = int(cfg['training']['batch_size'])
train_ds, cand_ds = build_tf_datasets(train_df, business_df, batch_size, USE_IMAGE, img_df)
rank_ds = build_rank_ds(train_df, business_df, batch_size, USE_IMAGE, img_df)

train_ds.element_spec

## 5) Train Stage 1: Retrieval (Two-tower)


In [ ]:
from modeling.retrieval_model import YelpRetrievalModel

user_ids = train_df['user_id'].astype(str).unique().tolist()
business_ids = business_df['business_id'].astype(str).unique().tolist()

retrieval = YelpRetrievalModel(
    user_ids=user_ids,
    business_ids=business_ids,
    embedding_dim=int(cfg['model']['embedding_dim']),
    max_tokens=int(cfg['model']['max_tokens']),
    query_seq_len=int(cfg['model']['query_seq_len']),
    biz_seq_len=int(cfg['model']['biz_seq_len']),
    mlp_hidden=int(cfg['model']['mlp_hidden']),
    use_query_text=bool(cfg['features']['use_query_text']),
    use_biz_text=bool(cfg['features']['use_biz_text']),
    use_image_embeddings=USE_IMAGE,
    image_embedding_dim=int(cfg['features']['image_embedding_dim']),
    candidates_ds=None,
    top_k=int(cfg['retrieval']['top_k']),
)

retrieval.adapt_text(train_ds, cand_ds)
retrieval.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=float(cfg['training']['learning_rate'])))
retrieval.fit(train_ds, epochs=int(cfg['training']['epochs_retrieval']), verbose=1)


## 6) Train Stage 2: Ranking (Reranker)

The ranking model learns a feature-rich scoring function to re-rank top-N retrieved candidates.


In [ ]:
from modeling.ranking_model import YelpRankingModel

ranker = YelpRankingModel(
    user_ids=user_ids,
    business_ids=business_ids,
    embedding_dim=int(cfg['model']['embedding_dim']),
    max_tokens=int(cfg['model']['max_tokens']),
    query_seq_len=int(cfg['model']['query_seq_len']),
    biz_seq_len=int(cfg['model']['biz_seq_len']),
    mlp_hidden=int(cfg['model']['mlp_hidden']),
    use_query_text=bool(cfg['features']['use_query_text']),
    use_biz_text=bool(cfg['features']['use_biz_text']),
    use_image_embeddings=USE_IMAGE,
    image_embedding_dim=int(cfg['features']['image_embedding_dim']),
)

ranker.adapt_text(rank_ds, cand_ds)
ranker.compile(optimizer=tf.keras.optimizers.Adagrad(learning_rate=float(cfg['training']['learning_rate'])))
ranker.fit(rank_ds, epochs=int(cfg['training']['epochs_ranking']), verbose=1)


## 7) Build HybridEngine (retrieve → rerank)


In [ ]:
from modeling.hybrid_engine import HybridEngine

engine = HybridEngine(
    retrieval_model=retrieval,
    business_df=business_df,
    use_biz_text=bool(cfg['features']['use_biz_text']),
    use_image_embeddings=USE_IMAGE,
    image_embeddings_df=img_df,
    top_k_retrieval=int(cfg['retrieval']['top_k']),
    ranking_model=ranker,
)

demo_user = str(train_df['user_id'].iloc[0])
demo_user

## 8) Demo queries


In [ ]:
engine.recommend(user_id=demo_user, query_text='', k=10)

In [ ]:
engine.recommend(user_id=demo_user, query_text='spicy ramen noodles', k=10)

In [ ]:
engine.recommend(user_id=demo_user, query_text='late night coffee', k=10)

## 9) Evaluation: Recall@K and NDCG@K


In [ ]:
from scripts.metrics import recall_at_k, ndcg_at_k
from tqdm import tqdm

k_eval = int(cfg['ranking']['top_k_eval'])
recalls, ndcgs = [], []

for row in tqdm(test_df.itertuples(index=False), total=len(test_df)):
    cand_ids = engine.retrieve(user_id=str(row.user_id), query_text='', k=int(cfg['retrieval']['top_k']))
    ranked = engine.rerank(user_id=str(row.user_id), query_text='', candidate_ids=cand_ids, k=k_eval)
    ranked_ids = ranked['business_id'].astype(str).tolist()
    recalls.append(recall_at_k(ranked_ids, str(row.business_id), k_eval))
    ndcgs.append(ndcg_at_k(ranked_ids, str(row.business_id), k_eval))

print(f'Users evaluated: {len(test_df)}')
print(f'Recall@{k_eval}: {np.mean(recalls):.4f}')
print(f'NDCG@{k_eval}:  {np.mean(ndcgs):.4f}')


## 10) Save models and outputs


In [ ]:
out_dir = cfg['experiment']['output_dir']
os.makedirs(out_dir, exist_ok=True)

retrieval.save(os.path.join(out_dir, 'retrieval_model'))
ranker.save(os.path.join(out_dir, 'ranking_model'))

demo = engine.recommend(user_id=demo_user, query_text='', k=10)
demo.to_csv(os.path.join(out_dir, 'demo_recommendations.csv'), index=False)
print('Saved to:', out_dir)